# Seaborn Interview EDA

Match distributions, category comparisons, relationships, and correlation plots to explicit analytical questions.

- **Study time:** 35-45 minutes
- **Prerequisites:** pandas summaries and basic plotting vocabulary
- **Mode:** `visual`
- **Data policy:** no downloads; deterministic synthetic customer data only; plot outputs are cleared after validation
- **Provenance:** rebuilt from the curated Seaborn tutorial around interview questions instead of a plot gallery

Output convention: every retained textual result begins with a label that identifies the operation that produced it.


In [ ]:
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

rng = np.random.default_rng(71)
sns.set_theme(style="whitegrid", context="notebook")


def show(label, value):
    print(f"\n--- {label} ---\n{value}")


n_rows = 500
segment = rng.choice(["new", "core", "premium"], size=n_rows, p=[0.35, 0.45, 0.20])
tenure = rng.integers(1, 73, size=n_rows)
base_spend = pd.Series(segment).map({"new": 45, "core": 80, "premium": 145}).to_numpy()
spend = base_spend + 0.8 * tenure + rng.normal(0, 25, size=n_rows)
support_calls = np.maximum(0, rng.poisson(2.5, size=n_rows) - (segment == "premium").astype(int))
churn_probability = 1 / (1 + np.exp(-(-1.5 - 0.015 * tenure + 0.25 * support_calls)))
customers = pd.DataFrame(
    {
        "segment": segment,
        "tenure_months": tenure,
        "monthly_spend": spend,
        "support_calls": support_calls,
        "churned": rng.random(n_rows) < churn_probability,
    }
)
show("Dataset | shape and columns", (customers.shape, customers.columns.tolist()))
show("Dataset | numeric summary", customers.describe().round(2).to_string())

## 1. Question: what is the spend distribution?


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(
    data=customers,
    x="monthly_spend",
    hue="segment",
    element="step",
    stat="density",
    common_norm=False,
    ax=ax,
)
ax.set(
    title="Monthly spend distribution by segment",
    xlabel="Monthly spend (currency units)",
    ylabel="Density",
)
fig.tight_layout()
plt.show()
plt.close(fig)

## 2. Question: how does spend vary by segment?


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=customers, x="segment", y="monthly_spend", order=["new", "core", "premium"], ax=ax)
sample = customers.sample(120, random_state=42)
sns.stripplot(
    data=sample,
    x="segment",
    y="monthly_spend",
    order=["new", "core", "premium"],
    color="black",
    alpha=0.35,
    size=3,
    ax=ax,
)
ax.set(title="Spend spread by customer segment", xlabel="Segment", ylabel="Monthly spend")
fig.tight_layout()
plt.show()
plt.close(fig)

## 3. Question: does tenure relate to spend differently by segment?


In [ ]:
sampled = customers.sample(250, random_state=42)
plot = sns.relplot(
    data=sampled,
    x="tenure_months",
    y="monthly_spend",
    hue="segment",
    col="churned",
    kind="scatter",
    alpha=0.65,
    height=4,
    aspect=1.0,
)
plot.set_axis_labels("Tenure (months)", "Monthly spend")
plot.figure.suptitle("Tenure and spend, faceted by churn outcome", y=1.04)
plt.show()
plt.close(plot.figure)

## 4. Question: which numeric variables move together?


In [ ]:
correlation = customers[["tenure_months", "monthly_spend", "support_calls", "churned"]].corr(
    numeric_only=True
)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="vlag", center=0, square=True, ax=ax)
ax.set_title("Selected numeric correlations")
fig.tight_layout()
plt.show()
plt.close(fig)

show("Correlation | matrix used by heatmap", correlation.round(3).to_string())

## 5. Interpretation discipline


In [ ]:
segment_summary = customers.groupby("segment", observed=True).agg(
    customers=("segment", "size"),
    median_spend=("monthly_spend", "median"),
    churn_rate=("churned", "mean"),
)
show("Interpretation | segment summary behind the plots", segment_summary.round(3).to_string())
show(
    "Visualization checks | status",
    "each plot has a question, labels, deterministic sampling, and an explicit numerical summary",
)